# Testes de Corretude

Verifica se **Backtracking** e **Meet in the Middle** produzem resultados corretos em relação ao oráculo (força bruta) para instâncias pequenas (n ≤ 14).

Coberturas:
- Casos de borda (conjunto vazio, alvo zero, um elemento)
- Instâncias geradas com solução
- Instâncias geradas sem solução
- Instâncias aleatórias pequenas
- Efeito da poda no backtracking

In [ ]:
import random
from utils import backtracking, forca_bruta, gerador, meet_in_middle

## Função de Verificação

Compara a decisão e o certificado dos dois algoritmos contra o oráculo (força bruta).

In [ ]:
def verificar_instancia(transacoes, alvo):
    esperado, _ = forca_bruta.resolver(transacoes, alvo)
    res_bt = backtracking.resolver(transacoes, alvo)
    res_mm = meet_in_middle.resolver(transacoes, alvo)

    assert res_bt.existe == esperado, (
        f"Backtracking divergiu do oráculo: {transacoes}, alvo={alvo}"
    )

    assert res_mm.existe == esperado, (
        f"MITM divergiu do oráculo: {transacoes}, alvo={alvo}"
    )

    if esperado:
        assert sum(res_bt.subconjunto) == alvo, "Certificado BT inválido"
        assert sum(res_mm.subconjunto) == alvo, "Certificado MITM inválido"
        assert all(transacoes[i] for i in res_bt.indices) or alvo == 0
        assert sorted(res_mm.subconjunto) == sorted(
            transacoes[i] for i in res_mm.indices
        )


def executar_suite(nome, casos):
    falhas = []

    for transacoes, alvo in casos:
        try:
            verificar_instancia(transacoes, alvo)
        except AssertionError as e:
            falhas.append(str(e))

    if falhas:
        print(f"FALHOU  {nome} ({len(falhas)} erro(s)):")
        for f in falhas:
            print(f"  • {f}")
    else:
        print(f"OK      {nome}")

    return len(falhas) == 0

## 1. Casos de Borda

In [ ]:
casos_borda = [
    ([], 0),          # conjunto vazio, alvo zero
    ([], 5),          # conjunto vazio, alvo positivo
    ([7], 7),         # um elemento, com solução
    ([7], 3),         # um elemento, sem solução
    ([5, 5, 5], 15),  # subconjunto = conjunto inteiro
    ([5, 5, 5], 0),   # alvo zero → subconjunto vazio
    ([2, 4, 6], 5),   # todos pares, alvo ímpar
]

for transacoes, alvo in casos_borda:
    esperado, cert_fb = forca_bruta.resolver(transacoes, alvo)
    res_bt = backtracking.resolver(transacoes, alvo)
    res_mm = meet_in_middle.resolver(transacoes, alvo)
    status = "OK" if res_bt.existe == esperado == res_mm.existe else "ERRO"
    print(
        f"{status}  transacoes={transacoes}, alvo={alvo} "
        f"→ esperado={esperado}, BT={res_bt.existe}, MITM={res_mm.existe}"
    )

executar_suite("casos_de_borda", casos_borda)

## 2. Instâncias Geradas com Solução

In [ ]:
casos_com_solucao = []
for n in range(2, 15):
    for semente in range(5):
        inst = gerador.gerar_com_solucao(n, semente)
        casos_com_solucao.append((inst.transacoes, inst.alvo))

print(f"Total de instâncias: {len(casos_com_solucao)}")
executar_suite("instancias_geradas_com_solucao", casos_com_solucao)

## 3. Instâncias Geradas sem Solução

In [ ]:
casos_sem_solucao = []
for n in range(2, 15):
    for semente in range(5):
        inst = gerador.gerar_sem_solucao(n, semente)
        casos_sem_solucao.append((inst.transacoes, inst.alvo))

print(f"Total de instâncias: {len(casos_sem_solucao)}")
executar_suite("instancias_geradas_sem_solucao", casos_sem_solucao)

## 4. Instâncias Aleatórias Pequenas

In [ ]:
rng = random.Random(2026)
casos_aleatorios = []
for _ in range(200):
    n = rng.randint(1, 12)
    transacoes = [rng.randint(0, 40) for _ in range(n)]
    alvo = rng.randint(0, sum(transacoes) + 10)
    casos_aleatorios.append((transacoes, alvo))

print(f"Total de instâncias: {len(casos_aleatorios)}")
executar_suite("instancias_aleatorias_pequenas", casos_aleatorios)

## 5. Poda Não Altera Resposta

Verifica que a poda do backtracking nunca muda a decisão: apenas reduz os nós visitados.

In [ ]:
rng = random.Random(7)
falhas_poda = []
economia_nos = []

for _ in range(100):
    n = rng.randint(1, 12)
    transacoes = [rng.randint(0, 40) for _ in range(n)]
    alvo = rng.randint(0, sum(transacoes) + 10)

    com_poda = backtracking.resolver(transacoes, alvo, usar_poda=True)
    sem_poda = backtracking.resolver(transacoes, alvo, usar_poda=False)

    if com_poda.existe != sem_poda.existe:
        falhas_poda.append(f"transacoes={transacoes}, alvo={alvo}")
    if com_poda.nos_visitados > sem_poda.nos_visitados:
        falhas_poda.append(
            f"Poda aumentou nós! {com_poda.nos_visitados} > {sem_poda.nos_visitados}"
        )

    economia = sem_poda.nos_visitados - com_poda.nos_visitados
    economia_nos.append(economia)

if falhas_poda:
    print(f"FALHOU  poda_nao_altera_resposta ({len(falhas_poda)} erro(s)):")
    for f in falhas_poda:
        print(f"  • {f}")
else:
    print("OK      poda_nao_altera_resposta")

print(f"\nEconomia média de nós por poda: {sum(economia_nos)/len(economia_nos):.1f}")
print(f"Economia máxima de nós:         {max(economia_nos)}")

## Resumo

In [ ]:
suites = [
    ("casos_de_borda",                  casos_borda),
    ("instancias_geradas_com_solucao",  casos_com_solucao),
    ("instancias_geradas_sem_solucao",  casos_sem_solucao),
    ("instancias_aleatorias_pequenas",  casos_aleatorios),
]

print("=" * 55)
print(f"{'Suite':<40} {'Instâncias':>10} {'Status':>6}")
print("=" * 55)

tudo_ok = True
for nome, casos in suites:
    ok = executar_suite(nome, casos)
    tudo_ok = tudo_ok and ok

# poda já foi verificada acima
ok_poda = len(falhas_poda) == 0
tudo_ok = tudo_ok and ok_poda
status_poda = "OK" if ok_poda else "FALHOU"
print(f"{status_poda}      poda_nao_altera_resposta")

print("=" * 55)
if tudo_ok:
    print("Todos os testes de corretude passaram.")
else:
    print("Há falhas: verifique as seções acima.")